In [ ]:
# Lab type: debug
# Course: DS202 — Time Series Analysis & Forecasting
# Lesson: Datetime Handling and Resampling in Pandas
# Task: The pipeline below contains 3 bugs. Each runs without error but silently
#       corrupts the weekly report it produces. Find each bug, explain it in the
#       markdown cell below it, and write the fixed version in the fix cell.

# Lab: Debugging a Weekly Orders Report

An analyst built a weekly orders report from a raw CSV export. The report runs
end-to-end with no errors — and three of its numbers are silently wrong.

For each bug:
1. Run the cell to see the symptom.
2. Identify the line that causes it.
3. Write your diagnosis in the markdown cell below.
4. Rewrite the fixed version in the fix cell.

**Outputs are cleared.** Run each cell to generate results.

## Setup: build the raw export

In [ ]:
!pip install pandas numpy scikit-learn statsmodels matplotlib --quiet

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
days = pd.date_range("2023-01-01", "2025-12-31", freq="D")
t = np.arange(len(days))

trend   = 200 + 0.15 * t
weekday = np.array([-14, -18, -11, -6, 9, 52, 61])[days.dayofweek]
yearly  = 38 * np.sin(2 * np.pi * (days.dayofyear - 320) / 365.25)
noise   = rng.normal(0, 16, len(days))

orders = pd.Series(trend + weekday + yearly + noise, index=days, name="orders").round()
print(f"{len(orders)} days, {orders.index[0].date()} to {orders.index[-1].date()}")
orders.head()

# Simulate the raw CSV export the analyst received:
#  - dates exported as strings in DAY-FIRST format (European system)
#  - a 9-day tracking outage: those rows are simply absent
outage = pd.date_range("2024-05-14", "2024-05-22", freq="D")
kept = orders.drop(outage)
raw = pd.DataFrame({
    "date": kept.index.strftime("%d/%m/%Y"),   # day-first strings!
    "orders": kept.values,
}).sample(frac=1.0, random_state=7).reset_index(drop=True)   # rows arrive unordered
raw.head()

## Bug 1: Parsing

The analyst parses the date column and immediately spot-checks January.

In [ ]:
# --- BUGGY CODE (Bug 1) ---
# Review this code — is it correct?
df = raw.copy()
df["date"] = pd.to_datetime(df["date"])          # ← no format specified
series = df.set_index("date")["orders"].sort_index()

# Symptom: how many days land in each of the first few months?
print(series.groupby(series.index.month).size().head(12))
print(f"\nrows parsed: {len(series)}  (expected 1087)")
print(f"days in January 2023: {len(series['2023-01'])}  (expected 31)")

**Explain the bug:** The row count is right, but the monthly distribution is wrong.
What did `pd.to_datetime` do to a string like `"05/03/2023"`? Which rows were affected
and which were parsed correctly, and why is that mixture worse than a uniform error?

*(Write your diagnosis here.)*

<details>
<summary>🔑 Reveal answer — Bug 1</summary>

**The bug:** The export is day-first, but `pd.to_datetime` without a `format` parses
ambiguous strings month-first. `"05/03/2023"` (5 March) became May 3.

**Why it's worse than a uniform error:** Only ambiguous dates (day ≤ 12) get swapped;
strings like `"25/04/2023"` can only be day-first and parse correctly. So roughly a
third of the rows silently moved to the wrong month while the rest stayed put — the
series still has the right length and plausible values, but weekday and month structure
is scrambled for the swapped rows.

**Correct approach:** Always pass an explicit format:
`pd.to_datetime(df["date"], format="%d/%m/%Y")`. A wrong format then raises a
`ValueError` instead of misreading.

</details>

In [ ]:
# Fix for Bug 1: explicit day-first format
df = raw.copy()
df["date"] = pd.to_datetime(df["date"], format="%d/%m/%Y")
series = df.set_index("date")["orders"].sort_index()

print(f"days in January 2023: {len(series['2023-01'])}  (expected 31)")
print(f"lag-7 autocorrelation: {series.autocorr(7):.3f}  (should be ≈ 0.93 — weekly structure restored)")

## Bug 2: The invisible gap

With dates fixed, the analyst builds the weekly totals for the Q2 2024 report.

In [ ]:
# --- BUGGY CODE (Bug 2) ---
# Review this code — is it correct?
weekly_totals = series.resample("W").sum()

print(weekly_totals["2024-05-05":"2024-05-28"])
print("\nAny NaN in the report?", weekly_totals.isna().any())

**Explain the bug:** The report shows a catastrophic week of 231 orders and a weak week
of 1,170 — but the store had a tracking outage, not a sales collapse. Why did `sum()`
produce numbers instead of NaNs, and what should the analyst have done *before*
resampling?

*(Write your diagnosis here.)*

<details>
<summary>🔑 Reveal answer — Bug 2</summary>

**The bug:** The nine outage days are *absent rows*, not NaNs — so `resample("W").sum()`
adds up whatever rows exist in each bucket and treats missing days as contributing
zero. No NaN, no warning, just an undercount that looks like a sales collapse.

**Correct approach:** Enforce the calendar grid first with `series.asfreq("D")` so the
gap becomes explicit NaNs, decide deliberately how to handle them, and guard the
aggregation with `sum(min_count=7)` (or at least `min_count=1`) so incomplete weeks
report NaN instead of a fictitious total.

</details>

In [ ]:
# Fix for Bug 2: surface the gap, then aggregate with a guard
daily = series.asfreq("D")
print(f"missing days now visible: {daily.isna().sum()}")

weekly_fixed = daily.resample("W").sum(min_count=7)   # NaN unless the week is complete
print(weekly_fixed["2024-05-05":"2024-05-28"])

## Bug 3: Filling the gap for a forecasting feature

The analyst then fills the gap so a downstream forecasting model gets a complete
series, and adds a smoothed feature.

In [ ]:
# --- BUGGY CODE (Bug 3) ---
# Review this feature prep — both filled columns feed a forecasting model.
model_input = pd.DataFrame({"orders": daily})
model_input["orders_filled"] = daily.interpolate()                       # fill the outage
model_input["orders_smooth"] = daily.interpolate().rolling(7, center=True).mean()

print(model_input["2024-05-12":"2024-05-24"].round(1))

**Explain the bug:** Both engineered columns contain temporal leakage, in two
different ways. For each column, identify exactly which future observations reach
which rows. (Hint from the lessons: what does linear interpolation connect? What
window does a centred rolling mean use?)

*(Write your diagnosis here.)*

<details>
<summary>🔑 Reveal answer — Bug 3</summary>

**The bug (two leaks):**

1. `interpolate()` fills the 14–22 May gap by drawing a line from 13 May to **23 May** —
   so every filled value inside the gap contains the observation from after the gap.
   As a model feature at, say, 15 May, that's future data.
2. `rolling(7, center=True)` computes each day's value from the window *t−3 … t+3* —
   three days of future for **every** row, not just the gap.

**Why it matters:** Both columns make training rows smarter than any live prediction
row can be, inflating evaluation and degrading silently in production.

**Correct approach:** For forecasting features, fill gaps with past-only methods
(`ffill()`), and build smoothed features with trailing windows shifted past the label:
`shift(1).rolling(7).mean()` (Lesson 6 makes this the standard pattern).

</details>

In [ ]:
# Fix for Bug 3: past-only fill, trailing (shifted) window
model_input = pd.DataFrame({"orders": daily})
model_input["orders_filled"] = daily.ffill()                       # repeats last known value
model_input["orders_smooth"] = daily.ffill().shift(1).rolling(7).mean()   # window t-7 … t-1

print(model_input["2024-05-12":"2024-05-24"].round(1))

## Summary

> **For each bug, complete the sentence in one line.**

1. **Parsing:** Without an explicit `format`, ambiguous day-first strings were parsed as ________.
2. **Resampling:** `resample("W").sum()` treated absent rows as ________, because the gap was ________ rather than NaN.
3. **Filling:** `interpolate()` and `center=True` windows both leak because they read values from ________.

<details>
<summary>🔑 Reveal summary answers</summary>

1. Ambiguous day-first strings were parsed as **month-first dates, silently swapping day
   and month for every date with day ≤ 12**.
2. `sum()` treated absent rows as **zero orders**, because the gap was **missing rows
   (invisible to the aggregation)** rather than NaN.
3. Both leak because they read values from **after the row being filled/smoothed — the
   future at prediction time**.

</details>